In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re
import spacy
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
# 1. Configuração do Spacy para INGLÊS
try:
    # Tenta carregar o modelo em inglês
    spc_en = spacy.load('en_core_web_sm')
except OSError:
    # Se não tiver, faz o download pelo sistema
    import os
    os.system('python -m spacy download en_core_web_sm')
    spc_en = spacy.load('en_core_web_sm')

# 2. Captura das Músicas (One Direction)
print("1/4: Iniciando a captura das músicas do One Direction (Aguarde)...")
page = requests.get("https://www.vagalume.com.br/one-direction/")
soup = BeautifulSoup(page.content, 'html.parser')
lista_alfabetica = BeautifulSoup(str(soup.find_all(id="alfabetMusicList")), 'html.parser')
a_tag = lista_alfabetica.find_all('a')

musicas = []
for a in a_tag:
    nome_musica = a.text.strip()
    if not(nome_musica == 'TRADUÇÃO' or nome_musica == ''):
        link_musica = a['href']
        musicas.append([nome_musica, link_musica])

for i in range(len(musicas)):
    link = "https://www.vagalume.com.br" + str(musicas[i][1])
    page = requests.get(link)
    soup = BeautifulSoup(page.content, 'html.parser')
    h3_tag = soup.find_all('h3')
    album = h3_tag[0].text if len(h3_tag) != 0 else ''

    lyrics = soup.find_all(id='lyrics')
    if len(lyrics) > 0:
        lyrics = str(lyrics[0])
        lyrics = lyrics.replace('<div id="lyrics">', '')
        lyrics = lyrics.replace('<div data-plugin="googleTranslate" id="lyrics">', '')
        lyrics = lyrics.replace('<br/>', ' ')
        lyrics = lyrics.replace("\'","'")
        lyrics = lyrics.replace('</div>', '')
    else:
        lyrics = ''

    musicas[i].append(album)
    musicas[i].append(lyrics)

musicas = pd.DataFrame(musicas, columns=['Nome da Música', 'link', 'album', 'letra'])

1/4: Iniciando a captura das músicas do One Direction (Aguarde)...


In [ ]:
# 3. Lista de Stopwords em Inglês (Nativa do Spacy + Customizadas)
print("2/4: Carregando as stopwords em inglês...")
# Pega as stopwords padrão do idioma inglês do Spacy
stopwords_en_set = spc_en.Defaults.stop_words

# Adicionamos alguns "ruídos" comuns em letras de música pop internacional
ruidos_musicais = {'oh', 'yeah', 'na', 'ooh', 'la', 'hey', 'ah', 'chorus', 'verse', 'woo', 'uh'}
stopwords_en_set = stopwords_en_set.union(ruidos_musicais)

# 4. Função de Pré-processamento
def limpar_texto_ingles(texto):
    # Regex adaptado para o alfabeto sem acentos (típico do inglês)
    letras = re.findall(r'\b[a-z]+\b', str(texto).lower())

    # Aplicando as stopwords
    meaningful_words = [w for w in letras if w not in stopwords_en_set]
    meaningful_words_string = " ".join(meaningful_words)

    # Prevenindo erro de string vazia no Spacy
    if not meaningful_words_string:
        return ""

    # Instanciando o objeto Spacy em INGLÊS
    spc_letras = spc_en(meaningful_words_string)

    # Lematização e Tokenização usando a Spacy
    # Pega o radical do verbo em inglês (ex: 'running' vira 'run')
    tokens = [token.lemma_ if token.pos_=='VERB' else str(token) for token in spc_letras]

    # Transforma a lista de tokens de volta em uma String para o TF-IDF
    return " ".join(tokens)

print("3/4: Aplicando pré-processamento nas músicas (Isso leva alguns segundos)...")
# Atualiza o dataframe
musicas['letra_processada'] = musicas['letra'].apply(limpar_texto_ingles)

# Remove músicas vazias
musicas = musicas[musicas['letra_processada'].str.strip() != ""]

2/4: Carregando as stopwords em inglês...
3/4: Aplicando pré-processamento nas músicas (Isso leva alguns segundos)...


In [ ]:
# 5. Cálculo do TF-IDF
print("4/4: Calculando TF-IDF e gerando a tabela...")
# O scikit-learn também tem uma lista nativa de stopwords em inglês, ativamos para garantir
vectorizer = TfidfVectorizer(stop_words='english')
tfidf_matrix = vectorizer.fit_transform(musicas['letra_processada'])

# Gerando a tabela formatada
df_tfidf = pd.DataFrame(
    data=tfidf_matrix.toarray(),
    index=musicas['Nome da Música'],
    columns=vectorizer.get_feature_names_out()
)

# Soma o TF-IDF de cada palavra em todas as músicas e pega as 10 maiores
palavras_mais_importantes = df_tfidf.sum().sort_values(ascending=False).head(10)

print("As 10 palavras que mais definem as músicas do One Direction:")
print(palavras_mais_importantes)

# Para ver o Top 5 palavras mais importantes da música 'Act My Age'
musica = 'Act My Age'
top_palavras = df_tfidf.loc[musica].sort_values(ascending=False).head(5)
print(top_palavras)

# Exibindo o resultado final
print("\n### Tabela TF-IDF: One Direction ###")
display(df_tfidf.round(4))


4/4: Calculando TF-IDF e gerando a tabela...
As 10 palavras que mais definem as músicas do One Direction:
know     9.232627
don      8.517311
love     8.108525
baby     7.126971
like     6.895456
ll       6.403998
want     6.101872
wanna    5.727402
let      5.487686
feel     5.157432
dtype: float64
act    0.637937
age    0.569381
win    0.233426
ll     0.185193
won    0.157267
Name: Act My Age, dtype: float64

### Tabela TF-IDF: One Direction ###


,accesories,act,actions,add,addict,addicted,addison,admit,adore,adventure,...,ya,yah,yeahh,years,yes,yesterday,young,younger,youth,zayn
Nome da Música,,,,,,,,,,,,,,,,,,,,,
18,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
A.M,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
Act My Age,0.0,0.6379,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
Alive,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
All You Need Is Love,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Wonderwall,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
You & I,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
You Are So Beautiful,0.0,0.0000,0.0,0.0000,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0000,0.0,0.0,0.0
